# COVID-19 Data Analysis Dashboard 📊

This notebook demonstrates comprehensive COVID-19 data analysis using real-time data from Johns Hopkins University.

## Learning Objectives
- Data cleaning and preprocessing
- Time series analysis
- Interactive visualizations
- Statistical modeling
- Dashboard creation

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 1. Data Loading and Preprocessing

In [ ]:
# Load COVID-19 data from Johns Hopkins University
def load_covid_data():
    """Load COVID-19 data from Johns Hopkins GitHub repository"""
    base_url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/"
    
    # Load confirmed cases
    confirmed_url = base_url + "time_series_covid19_confirmed_global.csv"
    confirmed_df = pd.read_csv(confirmed_url)
    
    # Load deaths
    deaths_url = base_url + "time_series_covid19_deaths_global.csv"
    deaths_df = pd.read_csv(deaths_url)
    
    # Load recoveries (if available)
    try:
        recovered_url = base_url + "time_series_covid19_recovered_global.csv"
        recovered_df = pd.read_csv(recovered_url)
    except:
        recovered_df = None
    
    return confirmed_df, deaths_df, recovered_df

# Load data
confirmed_df, deaths_df, recovered_df = load_covid_data()
print(f"✅ Data loaded successfully!")
print(f"📊 Confirmed cases shape: {confirmed_df.shape}")
print(f"💀 Deaths shape: {deaths_df.shape}")

In [ ]:
# Data preprocessing function
def preprocess_covid_data(df, metric_name):
    """Preprocess COVID-19 data for analysis"""
    # Melt the dataframe to convert dates to rows
    df_melted = df.melt(
        id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
        var_name='Date',
        value_name=metric_name
    )
    
    # Convert date column
    df_melted['Date'] = pd.to_datetime(df_melted['Date'])
    
    # Fill missing values
    df_melted[metric_name] = df_melted[metric_name].fillna(0)
    
    # Aggregate by country and date
    df_aggregated = df_melted.groupby(['Country/Region', 'Date'])[metric_name].sum().reset_index()
    
    return df_aggregated

# Preprocess data
confirmed_processed = preprocess_covid_data(confirmed_df, 'Confirmed')
deaths_processed = preprocess_covid_data(deaths_df, 'Deaths')

print("✅ Data preprocessing completed!")

## 2. Global Overview Analysis

In [ ]:
# Get latest global statistics
def get_global_stats(confirmed_df, deaths_df):
    """Calculate global COVID-19 statistics"""
    latest_date = confirmed_df['Date'].max()
    
    # Latest global numbers
    total_confirmed = confirmed_df[confirmed_df['Date'] == latest_date]['Confirmed'].sum()
    total_deaths = deaths_df[deaths_df['Date'] == latest_date]['Deaths'].sum()
    
    # Calculate daily new cases
    prev_date = latest_date - timedelta(days=1)
    new_confirmed = total_confirmed - confirmed_df[confirmed_df['Date'] == prev_date]['Confirmed'].sum()
    new_deaths = total_deaths - deaths_df[deaths_df['Date'] == prev_date]['Deaths'].sum()
    
    return {
        'date': latest_date,
        'total_confirmed': total_confirmed,
        'total_deaths': total_deaths,
        'new_confirmed': new_confirmed,
        'new_deaths': new_deaths,
        'mortality_rate': (total_deaths / total_confirmed) * 100 if total_confirmed > 0 else 0
    }

# Get global stats
global_stats = get_global_stats(confirmed_processed, deaths_processed)

print(f"📅 Date: {global_stats['date'].strftime('%B %d, %Y')}")
print(f"🌍 Total Confirmed Cases: {global_stats['total_confirmed']:,}")
print(f"💀 Total Deaths: {global_stats['total_deaths']:,}")
print(f"📈 New Cases Today: {global_stats['new_confirmed']:,}")
print(f"📉 New Deaths Today: {global_stats['new_deaths']:,}")
print(f"📊 Mortality Rate: {global_stats['mortality_rate']:.2f}%")

## 3. Interactive Visualizations

In [ ]:
# Create global trend visualization
def create_global_trends_chart(confirmed_df, deaths_df):
    """Create interactive global trends chart"""
    # Aggregate global data by date
    global_confirmed = confirmed_df.groupby('Date')['Confirmed'].sum().reset_index()
    global_deaths = deaths_df.groupby('Date')['Deaths'].sum().reset_index()
    
    # Calculate daily new cases and deaths
    global_confirmed['New_Cases'] = global_confirmed['Confirmed'].diff()
    global_deaths['New_Deaths'] = global_deaths['Deaths'].diff()
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Total Confirmed Cases', 'Total Deaths', 'Daily New Cases', 'Daily New Deaths'),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Add traces
    fig.add_trace(
        go.Scatter(x=global_confirmed['Date'], y=global_confirmed['Confirmed'],
                   mode='lines', name='Confirmed Cases', line=dict(color='blue')),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=global_deaths['Date'], y=global_deaths['Deaths'],
                   mode='lines', name='Deaths', line=dict(color='red')),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Bar(x=global_confirmed['Date'], y=global_confirmed['New_Cases'],
               name='New Cases', marker_color='lightblue'),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Bar(x=global_deaths['Date'], y=global_deaths['New_Deaths'],
               name='New Deaths', marker_color='lightcoral'),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text="Global COVID-19 Trends",
        height=800,
        showlegend=True
    )
    
    return fig

# Create and display the chart
global_trends_fig = create_global_trends_chart(confirmed_processed, deaths_processed)
global_trends_fig.show()

## 4. Country-wise Analysis

In [ ]:
# Top 10 countries by confirmed cases
def get_top_countries(df, n=10):
    """Get top N countries by confirmed cases"""
    latest_date = df['Date'].max()
    latest_data = df[df['Date'] == latest_date]
    
    top_countries = latest_data.nlargest(n, 'Confirmed')
    return top_countries

# Get top countries
top_countries = get_top_countries(confirmed_processed)

# Create bar chart
fig = px.bar(
    top_countries,
    x='Country/Region',
    y='Confirmed',
    title='Top 10 Countries by Confirmed COVID-19 Cases',
    color='Confirmed',
    color_continuous_scale='Reds'
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=500
)

fig.show()

## 5. Statistical Analysis

In [ ]:
# Calculate growth rates and statistics
def calculate_growth_metrics(df, country='Global'):
    """Calculate growth metrics for a country or globally"""
    if country == 'Global':
        data = df.groupby('Date')['Confirmed'].sum().reset_index()
    else:
        data = df[df['Country/Region'] == country].copy()
    
    # Calculate daily growth rate
    data['Daily_Growth_Rate'] = data['Confirmed'].pct_change() * 100
    
    # Calculate 7-day moving average
    data['MA7'] = data['Confirmed'].rolling(window=7).mean()
    
    # Calculate doubling time (simplified)
    data['Doubling_Time'] = np.log(2) / np.log(1 + data['Daily_Growth_Rate']/100)
    
    return data

# Calculate metrics for top countries
growth_analysis = {}
for country in top_countries['Country/Region'].head(5):
    growth_analysis[country] = calculate_growth_metrics(confirmed_processed, country)

print("📊 Growth Analysis for Top 5 Countries:")
for country, data in growth_analysis.items():
    latest_growth = data['Daily_Growth_Rate'].iloc[-1]
    latest_doubling = data['Doubling_Time'].iloc[-1]
    print(f"{country}: Growth Rate: {latest_growth:.2f}%, Doubling Time: {latest_doubling:.1f} days")

## 6. Key Insights and Conclusions

### 📈 **Key Findings:**
1. **Global Impact**: COVID-19 has affected every continent with varying severity
2. **Growth Patterns**: Different countries show different growth trajectories
3. **Mortality Rates**: Vary significantly between countries due to healthcare capacity and demographics
4. **Recovery Patterns**: Some countries show signs of recovery while others continue to struggle

### 🎯 **Learning Outcomes:**
- ✅ Data cleaning and preprocessing techniques
- ✅ Time series analysis and visualization
- ✅ Statistical modeling and growth rate calculations
- ✅ Interactive dashboard creation
- ✅ Real-world data science workflow

### 🚀 **Next Steps:**
- Build predictive models for case forecasting
- Create interactive maps with Folium
- Develop a Streamlit dashboard
- Add vaccination data analysis
- Implement machine learning for trend prediction